In [1]:
import numpy
import pandas
import six
import tensorflow as tf

import CS230

In [2]:
file_paths = CS230.data.get_all_file_paths()

# experiment #1
- input: steering/brake/throttle
- output: discrete derivative of all vehicle movement

# load data sets

In [107]:
file_path = file_paths[0]

train_percent = 0.9
dev_percent = 0.05
test_percent = 0.05

df = CS230.data.load(file_path)
df = CS230.data.stride_rows(df, stride=10)
df = CS230.data.add_derivatives(df, stride=1)
df = CS230.data.clean_discontinuities(df, stride=1)

df_train, df_dev, df_test = CS230.data.get_data_sets(df, train_percent, dev_percent, test_percent, 
                                                     CS230.data.COLUMNS_HUMAN_INPUT, CS230.data.COLUMNS_DERIV)

# define model

In [108]:
model = tf.keras.Sequential()

model.add(tf.keras.layers.Dense(len(CS230.data.COLUMNS_HUMAN_INPUT), activation='relu'))

model.add(tf.keras.layers.Dense(len(CS230.data.COLUMNS_DERIV), activation='relu'))

model.add(tf.keras.layers.Dropout(0.2))

model.add(tf.keras.layers.Dense(len(CS230.data.COLUMNS_DERIV), activation='sigmoid'))

In [109]:
model.compile(
    optimizer=tf.train.AdamOptimizer(0.001), 
    loss=tf.keras.losses.MeanSquaredError(), 
    metrics=['accuracy']
)

# view training data set

In [110]:
df_train.head()

,brake,throttle,handwheelAngle,deriv_axCG,deriv_ayCG,deriv_pitchAngle,deriv_pitchRate,deriv_rollAngle,deriv_rollRate,deriv_vxCG,deriv_vyCG,deriv_wheelAccelFL,deriv_wheelAccelFR,deriv_wheelAccelRL,deriv_wheelAccelRR,deriv_yawAngle,deriv_yawRate
200625,0.0,55.2,24.7,-0.57,-3.02,0.03,1.84,-0.05,-5.38,0.03,-0.01,1.47,0.59,-6.08,-8.29,0.14,-0.36
249487,0.0,0.9,-5.6,0.01,0.02,0.00,0.58,0.00,-0.07,0.00,0.00,-0.10,-0.10,0.20,-0.07,0.00,0.09
60591,0.0,0.8,-33.1,0.05,-0.04,0.00,-0.26,0.00,0.35,0.00,0.00,0.19,-0.10,0.20,0.01,0.01,0.65
201647,0.0,87.9,1.5,-3.68,-8.91,0.00,-5.60,-0.05,-2.78,0.00,-0.11,-12.36,-1.37,-0.98,4.13,-0.04,0.90
95006,13.5,2.2,-34.2,-0.21,1.01,0.07,1.62,-0.04,0.10,0.00,0.01,1.47,-1.47,-0.29,-0.62,0.00,-0.15


In [111]:
x_train = tf.cast(df_train[CS230.data.COLUMNS_HUMAN_INPUT].values, tf.float32)
y_train = tf.cast(df_train[CS230.data.COLUMNS_DERIV].values, tf.float32)

In [112]:
df_train[CS230.data.COLUMNS_HUMAN_INPUT].values

array([[  0. ,  55.2,  24.7],
       [  0. ,   0.9,  -5.6],
       [  0. ,   0.8, -33.1],
       ...,
       [  0. ,   0.9,  -7.1],
       [  0. ,   0.7, -33.7],
       [  0. ,   0.8, -33.1]])

In [113]:
df_train[CS230.data.COLUMNS_DERIV].values

array([[-0.57, -3.02,  0.03, ..., -8.29,  0.14, -0.36],
       [ 0.01,  0.02,  0.  , ..., -0.07,  0.  ,  0.09],
       [ 0.05, -0.04,  0.  , ...,  0.01,  0.01,  0.65],
       ...,
       [-0.02,  0.04, -0.01, ...,  0.11,  0.  ,  0.14],
       [ 0.03, -0.03,  0.01, ..., -0.08,  0.  , -0.17],
       [ 0.04,  0.03,  0.  , ..., -0.33,  0.  , -0.13]])

# train model

In [115]:
model.fit(x_train, y_train, epochs=10, steps_per_epoch=500)

Epoch 1/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0576 - acc: 0.0761
Epoch 2/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0554 - acc: 0.0819
Epoch 3/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0546 - acc: 0.0993
Epoch 4/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0543 - acc: 0.1079
Epoch 5/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0541 - acc: 0.1083
Epoch 6/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0540 - acc: 0.1091
Epoch 7/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0539 - acc: 0.1101
Epoch 8/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0539 - acc: 0.1116
Epoch 9/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0538 - acc: 0.1137
Epoch 10/10
500/500 [==============================] - 10s 20ms/step - loss: 17.0538 - acc: 0.1166


# view dev set

In [116]:
df_dev.head()

,brake,throttle,handwheelAngle,deriv_axCG,deriv_ayCG,deriv_pitchAngle,deriv_pitchRate,deriv_rollAngle,deriv_rollRate,deriv_vxCG,deriv_vyCG,deriv_wheelAccelFL,deriv_wheelAccelFR,deriv_wheelAccelRL,deriv_wheelAccelRR,deriv_yawAngle,deriv_yawRate
131343,7.9,0.7,1.3,1.50,0.77,-0.02,0.44,0.18,12.37,-0.05,-0.02,10.60,3.93,-7.56,-33.05,0.03,0.17
191633,0.0,34.5,-38.9,5.97,-1.88,-0.20,5.63,0.22,2.09,0.08,0.04,-1.28,5.20,8.34,17.93,-0.22,-0.63
66438,0.0,0.8,-33.1,0.03,-0.03,0.00,0.06,-0.01,0.02,0.00,0.00,0.19,0.09,0.10,0.18,0.00,0.21
148370,0.0,45.9,-11.5,1.00,5.77,0.00,4.71,0.02,-2.22,0.05,0.05,18.93,-13.83,-11.77,4.21,-0.08,-0.51
150370,0.0,71.3,-11.8,0.10,-3.12,0.02,3.13,-0.05,2.00,0.02,-0.04,12.06,1.86,-1.57,0.45,-0.08,0.91


In [117]:
x_dev = tf.cast(df_dev[CS230.data.COLUMNS_HUMAN_INPUT].values, tf.float32)
y_dev = tf.cast(df_dev[CS230.data.COLUMNS_DERIV].values, tf.float32)

In [118]:
df_dev[CS230.data.COLUMNS_HUMAN_INPUT].values

array([[  7.9,   0.7,   1.3],
       [  0. ,  34.5, -38.9],
       [  0. ,   0.8, -33.1],
       ...,
       [  0. ,   0.7,  -7.9],
       [  0. ,  51.6,  31.7],
       [  0. ,  23.4, -25.6]])

In [119]:
df_dev[CS230.data.COLUMNS_DERIV].values

array([[ 1.500e+00,  7.700e-01, -2.000e-02, ..., -3.305e+01,  3.000e-02,
         1.700e-01],
       [ 5.970e+00, -1.880e+00, -2.000e-01, ...,  1.793e+01, -2.200e-01,
        -6.300e-01],
       [ 3.000e-02, -3.000e-02,  0.000e+00, ...,  1.800e-01,  0.000e+00,
         2.100e-01],
       ...,
       [ 8.600e-01, -5.200e-01,  2.000e-02, ...,  2.740e+00, -2.000e-02,
         2.800e-01],
       [ 2.460e+00,  3.040e+00,  3.000e-02, ...,  1.217e+01,  5.000e-02,
        -5.900e-01],
       [-3.050e+00, -4.710e+00,  9.000e-02, ...,  2.950e+00, -1.300e-01,
        -4.600e-01]])

# predict against dev set

In [120]:
dev_predictions = model.predict(x_dev, steps=1)

In [121]:
dev_predictions

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [3.13457847e-03, 3.87430191e-07, 1.15656853e-03, ...,
        2.11596489e-06, 7.06315041e-05, 1.21788979e-02],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 8.94069672e-08],
       ...,
       [9.43809748e-04, 1.19209290e-07, 3.54468822e-04, ...,
        1.78813934e-07, 1.40964985e-05, 6.09290600e-03],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 8.94069672e-08]], dtype=float32)

In [122]:
df_dev_predictions = pandas.DataFrame(dev_predictions, columns=CS230.data.COLUMNS_DERIV)

In [123]:
df_dev.head()

,brake,throttle,handwheelAngle,deriv_axCG,deriv_ayCG,deriv_pitchAngle,deriv_pitchRate,deriv_rollAngle,deriv_rollRate,deriv_vxCG,deriv_vyCG,deriv_wheelAccelFL,deriv_wheelAccelFR,deriv_wheelAccelRL,deriv_wheelAccelRR,deriv_yawAngle,deriv_yawRate
131343,7.9,0.7,1.3,1.50,0.77,-0.02,0.44,0.18,12.37,-0.05,-0.02,10.60,3.93,-7.56,-33.05,0.03,0.17
191633,0.0,34.5,-38.9,5.97,-1.88,-0.20,5.63,0.22,2.09,0.08,0.04,-1.28,5.20,8.34,17.93,-0.22,-0.63
66438,0.0,0.8,-33.1,0.03,-0.03,0.00,0.06,-0.01,0.02,0.00,0.00,0.19,0.09,0.10,0.18,0.00,0.21
148370,0.0,45.9,-11.5,1.00,5.77,0.00,4.71,0.02,-2.22,0.05,0.05,18.93,-13.83,-11.77,4.21,-0.08,-0.51
150370,0.0,71.3,-11.8,0.10,-3.12,0.02,3.13,-0.05,2.00,0.02,-0.04,12.06,1.86,-1.57,0.45,-0.08,0.91


In [124]:
df_dev_predictions.head()

,deriv_axCG,deriv_ayCG,deriv_pitchAngle,deriv_pitchRate,deriv_rollAngle,deriv_rollRate,deriv_vxCG,deriv_vyCG,deriv_wheelAccelFL,deriv_wheelAccelFR,deriv_wheelAccelRL,deriv_wheelAccelRR,deriv_yawAngle,deriv_yawRate
0,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.001245,0.000000e+00,0.000000,0.000000,0.000000e+00
1,0.003135,3.874302e-07,0.001157,0.003999,0.004219,0.000132,0.012968,5.444258e-03,0.103639,0.000028,8.052707e-03,0.000002,0.000071,1.217890e-02
2,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.001583,0.000000,0.000000e+00,0.000000,0.000000,8.940697e-08
3,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00
4,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,4.172325e-07,0.010115,0.000000,5.364418e-07,0.000000,0.000000,2.086163e-07


# experiment #2
- input: steering/brake/throttle
- output: discrete derivative x/y velocity

In [125]:
file_path = file_paths[0]
data_columns = CS230.data.COLUMNS_HUMAN_INPUT
label_columns = ['deriv_vxCG', 'deriv_vyCG']

train_percent = 0.9
dev_percent = 0.05
test_percent = 0.05

df = CS230.data.load(file_path)
df = CS230.data.stride_rows(df, stride=10)
df = CS230.data.add_derivatives(df, stride=1)
df = CS230.data.clean_discontinuities(df, stride=1)

df_train, df_dev, df_test = CS230.data.get_data_sets(df, train_percent, dev_percent, test_percent, 
                                                     data_columns, label_columns)

In [132]:
model = tf.keras.Sequential()

model.add(tf.keras.layers.Dense(len(data_columns), activation='relu'))

model.add(tf.keras.layers.Dense(len(label_columns), activation='relu'))

model.add(tf.keras.layers.Dropout(0.2))

model.add(tf.keras.layers.Dense(len(label_columns), activation='sigmoid'))

model.compile(
    optimizer=tf.train.AdamOptimizer(0.001), 
    loss=tf.keras.losses.MeanSquaredError(), 
    metrics=['accuracy']
)

In [133]:
x_train = tf.cast(df_train[data_columns].values, tf.float32)
y_train = tf.cast(df_train[label_columns].values, tf.float32)

In [134]:
df_train[data_columns].values

array([[  0. ,   0.9,  -6.4],
       [  0. ,   0.7,  29.3],
       [  0. ,   0.8, -33.5],
       ...,
       [  0. ,   2. ,  29.8],
       [  0. ,   6.5,  -9.8],
       [  0. ,   0.7, -33. ]])

In [135]:
df_train[label_columns].values

array([[ 0.  ,  0.  ],
       [-0.  ,  0.  ],
       [ 0.  , -0.  ],
       ...,
       [-0.03,  0.02],
       [-0.01,  0.04],
       [ 0.  ,  0.  ]])

In [136]:
model.fit(x_train, y_train, epochs=10, steps_per_epoch=500)

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
Epoch 1/10
500/500 [==============================] - 6s 12ms/step - loss: 0.2000 - acc: 0.6948
Epoch 2/10
500/500 [==============================] - 6s 12ms/step - loss: 0.1170 - acc: 0.7038
Epoch 3/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0716 - acc: 0.7038
Epoch 4/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0458 - acc: 0.7038
Epoch 5/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0307 - acc: 0.7038
Epoch 6/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0214 - acc: 0.7038
Epoch 7/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0154 - acc: 0.7038
Epoch 8/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0114 - acc: 0.7038
Epoch 9/10
500/500 [==============================] - 6s 12ms/step - loss: 0.0086 - acc: 0.7038
Epoch 10/10
500/500 [

In [89]:
x_dev = tf.cast(df_dev[data_columns].values, tf.float32)
y_dev = tf.cast(df_dev[label_columns].values, tf.float32)

In [90]:
dev_predictions = model.predict(x_dev, steps=1)

In [92]:
df_dev.head()

,brake,throttle,handwheelAngle,deriv_vxCG,deriv_vyCG
228852,0.0,3.4,-75.8,-0.01,-0.01
227978,0.0,2.2,-182.4,0.00,0.00
103753,0.0,41.7,4.0,0.03,0.00
98055,0.0,1.0,250.4,0.00,0.00
190580,0.5,0.6,-13.6,-0.05,0.07


In [95]:
max(df_dev['deriv_vxCG'])

0.14000000000000057

In [96]:
max(df_dev['deriv_vyCG'])

0.2699999999999998

In [97]:
df_dev_predictions = pandas.DataFrame(dev_predictions, columns=label_columns)

In [98]:
df_dev_predictions.head()

,deriv_vxCG,deriv_vyCG
0,0.003573,0.000000
1,0.154065,0.000971
2,0.369859,0.365840
3,0.422075,0.239431
4,0.268553,0.042679
